<a href="https://colab.research.google.com/github/Prachii26/DeepLearningCMPE258/blob/main/Advanced%20customizations%20in%20deep%20learning%20and%20neural%20networks/Colabs/Advanced_customizations_in_DL_and_NN_Part2(WANDB).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q wandb
import wandb
wandb.login(key="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")
print("Login done!")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: prachigupta2610 (prachigupta2610-student) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Login done!


In [2]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import wandb

# Load data
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
x_train_full = x_train_full.astype("float32") / 255.0
x_train_full = x_train_full.reshape(-1, 784)
x_train, y_train = x_train_full[:5000], y_train_full[:5000]
x_val,   y_val   = x_train_full[5000:6000], y_train_full[5000:6000]

# Init W&B
wandb.init(
    project = "fashion-mnist-demo",
    config  = {"lr": 0.001, "epochs": 5, "batch_size": 32}
)

# Build model
model = keras.Sequential([
    keras.Input(shape=(784,)),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(64,  activation="relu"),
    keras.layers.Dense(10,  activation="softmax")
])
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

# Train
h = model.fit(x_train, y_train, epochs=5, batch_size=32,
              validation_data=(x_val, y_val), verbose=1)

# Log to W&B
for i in range(5):
    wandb.log({
        "epoch"      : i + 1,
        "train_loss" : h.history["loss"][i],
        "train_acc"  : h.history["accuracy"][i],
        "val_acc"    : h.history["val_accuracy"][i]
    })

wandb.finish()
print(f"✅ Done! Val acc: {h.history['val_accuracy'][-1]:.3f}")

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.6509 - loss: 1.1036 - val_accuracy: 0.7690 - val_loss: 0.6808
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8090 - loss: 0.5594 - val_accuracy: 0.7930 - val_loss: 0.5699
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8356 - loss: 0.4800 - val_accuracy: 0.7630 - val_loss: 0.6773
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8425 - loss: 0.4402 - val_accuracy: 0.8160 - val_loss: 0.5363
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8747 - loss: 0.3709 - val_accuracy: 0.8110 - val_loss: 0.5390


epoch,▁▃▅▆█
train_acc,▁▅▆▇█
train_loss,█▄▃▂▁
val_acc,▂▅▁█▇
epoch,5
train_acc,0.8698
train_loss,0.37148
val_acc,0.811


✅ Done! Val acc: 0.811


In [3]:
import torch
import torch.nn as nn
import wandb

# Init new run
wandb.init(
    project = "fashion-mnist-pytorch",
    config  = {"lr": 0.001, "epochs": 5}
)

# Data
xt = torch.tensor(x_train, dtype=torch.float32)
yt = torch.tensor(y_train, dtype=torch.long)
xvt = torch.tensor(x_val,  dtype=torch.float32)
yvt = torch.tensor(y_val,  dtype=torch.long)
dl  = torch.utils.data.DataLoader(
      torch.utils.data.TensorDataset(xt, yt),
      batch_size=32, shuffle=True)

# Model
model_pt  = nn.Sequential(
    nn.Linear(784, 128), nn.ReLU(),
    nn.Linear(128, 64),  nn.ReLU(),
    nn.Linear(64, 10)
)
optimizer = torch.optim.Adam(model_pt.parameters(), lr=0.001)
loss_fn   = nn.CrossEntropyLoss()

# Train
for epoch in range(5):
    model_pt.train()
    total_loss = 0
    for xb, yb in dl:
        optimizer.zero_grad()
        loss = loss_fn(model_pt(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model_pt.eval()
    with torch.no_grad():
        val_acc = (model_pt(xvt).argmax(dim=1)==yvt).float().mean().item()

    wandb.log({
        "epoch"      : epoch + 1,
        "train_loss" : total_loss / len(dl),
        "val_acc"    : val_acc
    })
    print(f"Epoch {epoch+1} | Val Acc: {val_acc:.3f}")

wandb.finish()
print("✅ PyTorch Done!")

Epoch 1 | Val Acc: 0.747
Epoch 2 | Val Acc: 0.774
Epoch 3 | Val Acc: 0.793
Epoch 4 | Val Acc: 0.814
Epoch 5 | Val Acc: 0.817


epoch,▁▃▅▆█
train_loss,█▃▂▂▁
val_acc,▁▄▆██
epoch,5
train_loss,0.41824
val_acc,0.817


✅ PyTorch Done!
